# Understanding Parent and Child Runs in MLflow

## Introduction
Machine learning projects often involve intricate relationships. These connections can emerge at various stages, be it the project's conception, during data preprocessing, in the model's architecture, or even during the model's tuning process. MLflow provides tools to efficiently capture and represent these relationships.

## Core Concepts of MLflow: Tags, Experiments, and Runs
In our foundational MLflow tutorial, we highlighted a fundamental relationship: the association between tags, experiments, and runs. This association is crucial when dealing with complex ML projects, such as forecasting models for individual products in a supermarket, as presented in our example. The diagram below offers a visual representation:

![A_model_grouping_hierarchy](../../../../images/A_model_grouping_hierarchy.png)


### Key Aspects
* Tags: These are instrumental in defining business-level filtering keys. They aid in retrieving relevant experiments and their runs.

* Experiments: They set boundaries, both from a business perspective and data-wise. For instance, sales data for carrots wouldn't be used to predict sales of apples without prior validation.

* Runs: Each run captures a specific hypothesis or iteration of training, nestled within the context of the experiment.


## The Real-world Challenge: Hyperparameter Tuning
While the above model suffices for introductory purposes, real-world scenarios introduce complexities. One such complexity arises when tuning models.

Model tuning is paramount. Methods range from grid search (though typically not recommended due to inefficiencies) to random searches, and more advanced approaches like automated hyperparameter tuning. The objective remains the same: to optimally traverse the model's parameter space.

### Benefits of Hyperparameter Tuning
* Loss Metric Relationship: By analyzing the relationship between hyperparameters and optimization loss metrics, we can discern potentially irrelevant parameters.

* Parameter Space Analysis: Monitoring the range of tested values can indicate if we need to constrict or expand our search space.

* Model Sensitivity Analysis: Estimating how a model reacts to specific parameters can pinpoint potential feature set issues.

But here lies the challenge: How do we systematically store the extensive data produced during hyperparameter tuning?

![The_quandary_of_storing_hyperparameter_data](../../../../images/The_quandary_of_storing_hyperparameter_data.png)

### What are Parent and Child Runs?
At its core, MLflow allows users to track experiments, which are essentially named groups of runs. A "run" in this context refers to a single execution of a model training event, where you can log parameters, metrics, tags, and artifacts associated with the training process. The concept of Parent and Child Runs introduces a hierarchical structure to these runs.

Imagine a scenario where you're testing a deep learning model with different architectures. Each architecture can be considered a parent run, and every iteration of hyperparameter tuning for that architecture becomes a child run nested under its respective parent.

### Benefits
1. <b>Organizational Clarity:</b> By using Parent and Child Runs, you can easily group related runs together. For instance, if you're running a hyperparameter search using a Bayesian approach on a particular model architecture, every iteration can be logged as a child run, while the overarching Bayesian optimization process can be the parent run.

2. <b>Enhanced Traceability:</b> When working on large projects with a broad product hierarchy, child runs can represent individual products or variants, making it straightforward to trace back results, metrics, or artifacts to their specific run.

3. <b>Scalability:</b> As your experiments grow in number and complexity, having a nested structure ensures that your tracking remains scalable. It's much easier to navigate through a structured hierarchy than a flat list of hundreds or thousands of runs.

4. <b>Improved Collaboration:</b> For teams, this approach ensures that members can easily understand the structure and flow of experiments conducted by their peers, promoting collaboration and knowledge sharing.

### Relationship between Experiments, Parent Runs, and Child Runs
* <b>Experiments:</b> Consider experiments as the topmost layer. They are named entities under which all related runs reside. For instance, an experiment named "Deep Learning Architectures" might contain runs related to various architectures you're testing.

* <b>Parent Runs:</b> Within an experiment, a parent run represents a significant segment or phase of your workflow. Taking the earlier example, each specific architecture (like CNN, RNN, or Transformer) can be a parent run.

* <b>Child Runs:</b> Nested within parent runs are child runs. These are iterations or variations within the scope of their parent. For a CNN parent run, different sets of hyperparameters or slight architectural tweaks can each be a child run.

### Practical Example
For this example, let's image that we're working through a fine-tuning exercise for a particular modeling solution. We're going through the tuning phase of rough adjustments initially, attempting to determine which parameter ranges and categorical selection values that we might want to consider for a full hyperparameter tuning run with a much higher iteration count.

### Naive Approach with no child runs
In this first phase, we will be trying relatively small batches of different combinations of parameters and evaluating them within the MLflow UI to determine whether we should include or exempt certain values based on the relatively performance amongst our iterative trials.

If we were to use each iteration as its own MLflow run, our code might look something like this:

In [ ]:
import random
import mlflow
from functools import partial
from itertools import starmap
from more_itertools import consume


# Define a function to log parameters and metrics
def log_run(run_name, test_no):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("param1", random.choice(["a", "b", "c"]))
        mlflow.log_param("param2", random.choice(["d", "e", "f"]))
        mlflow.log_metric("metric1", random.uniform(0, 1))
        mlflow.log_metric("metric2", abs(random.gauss(5, 2.5)))


# Generate run names
def generate_run_names(test_no, num_runs=5):
    return (f"run_{i}_test_{test_no}" for i in range(num_runs))


# Execute tuning function
def execute_tuning(test_no):
    # Partial application of the log_run function
    log_current_run = partial(log_run, test_no=test_no)
    # Generate run names and apply log_current_run function to each run name
    runs = starmap(log_current_run, ((run_name,) for run_name in generate_run_names(test_no)))
    # Consume the iterator to execute the runs
    consume(runs)


# Set the tracking uri and experiment
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("No Child Runs")

# Execute 5 hyperparameter tuning runs
consume(starmap(execute_tuning, ((x,) for x in range(5))))